<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

Install NumPy. This notebook was tested on NumPy 1.26, where np.NINF still exists, because NumPy 2.0 removed that constant and the function below would raise AttributeError.

In [ ]:
!pip install numpy

Leave OpenBB out of the pip command. Installing it into an environment you already use makes pip re-resolve pinned pandas and plotting requirements, causing resolver failures or silent upgrades that break imports later, so I keep openbb_terminal in its own conda environment on Python 3.10.

## Imports and setup

NumPy does the array math that filters and squares the returns, while the OpenBB SDK handles the price download so you don't write authentication or HTTP request code yourself.

In [ ]:
import numpy as np

In [ ]:
import yfinance as yf

## Load AAPL prices and compute returns

Pull AAPL daily bars from Yahoo Finance, the default source for this call, then turn the adjusted closes into daily percent changes, checking returns.index.min() and len(returns) to see what you actually got since the default start date moves with today's date.

In [ ]:
data = yf.download("AAPL", start="2020-01-01")
returns = data.Close.pct_change()

We use the autoadjusted close because it accounts for splits and dividends, so the 4-for-1 split in August 2020 doesn't show up as a 75% loss. The first value of returns is NaN since there's no prior day to compare against, which is why the function below uses nanmean instead of mean.

## Define the downside deviation function

This measures how far returns fall below a target of zero, treats every up day as zero instead of dropping it, then annualizes the result.

In [ ]:
def downside_deviation(returns):
    out = np.empty(returns.shape[1:])

    downside_diff = np.clip(returns, np.NINF, 0)
    np.square(downside_diff, out=downside_diff)
    np.nanmean(downside_diff, axis=0, out=out)
    np.sqrt(out, out=out)
    np.multiply(out, np.sqrt(252), out=out)

    return out.item()

Standard deviation subtracts the mean and squares every deviation. This function uses zero as the target and clips positive returns to 0, so a 6% up day adds 0 to the sum while a 6% down day adds 0.0036, and both days stay in the denominator. Multiplying by the square root of 252 annualizes it, using the 252 trading day convention for US equities, so it sits on the same scale as annualized volatility while counting only losses.

np.square, np.nanmean, np.sqrt, and np.multiply each take out= and write into arrays that already exist, though np.clip still allocates and .item() returns a Python float. On one ticker with a few thousand rows I can't measure the difference, so treat the pattern as preparation for running this across a large universe, not a speedup you'll see here.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.